In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')


np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from normal_evaluation.drbart_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'pcr'

n_processes = 32
batch_size = 50

log_name = 'test'

with open('../transformed_event_logs/PCR_start_end_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

#test_event_log['time:timestamp'] = test_event_log['time:timestamp_complete']
test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_activities = ['Callback timeout', 'Export result', 'Export to EMS', 'Match patient data', 'Receive sample state', 'Send notification', 'Wait for plate validation', 'timeout']

/tmp/ipykernel_3187476/2211210620.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_event_log = pickle.load(f)


In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

likelihoods_A = None
likelihoods_R = None
likelihoods_R_A = None
likelihoods_R_A_S = None
likelihoods_R_A_S_AC = None
likelihoods_R_A_S_RC = None
likelihoods_R_A_S_RC_AC = None
likelihoods_R_A_S_RC_AC_V = None
likelihoods_R_A_S_D = None
likelihoods_R_A_S_D_RC_AC = None

In [4]:
drbart_model_A = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name/',
                     strict_parser=False)
evaluator_A = conduct_evaluation.ConductEvaluation(drbart_model_A, SampleOutcomes_DRBART_Normal_A, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'resources' : False,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 1235/1235 [00:05<00:00, 246.02it/s]


In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.7423690904091179969847784570')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(40229.08503254297)

In [5]:
drbart_model_R_A_S = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day/',
                     strict_parser=False)
evaluator_R_A_S = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S, SampleOutcomes_DRBART_Normal,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'resources' : False,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S = evaluator_R_A_S.sample_cases(False, True)

100%|██████████| 1235/1235 [00:04<00:00, 264.16it/s]


In [6]:
np.mean([v.ln() for v in likelihoods_R_A_S[0].values()])

Decimal('-10.23686154151741184930226673')

In [7]:
np.mean(get_pscores(likelihoods_R_A_S))

np.float64(21296.666593207232)

In [10]:
drbart_model_R_A_S_AC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day_activity-count/',
                     strict_parser=False)
evaluator_R_A_S_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_AC, SampleOutcomes_DRBART_Normal,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)'                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'resources' : False,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_AC = evaluator_R_A_S_AC.sample_cases(False, True)

100%|██████████| 1235/1235 [00:04<00:00, 261.55it/s]


In [11]:
np.mean([v.ln() for v in likelihoods_R_A_S_AC[0].values()])

Decimal('-5.017866141304894305686983615')

In [12]:
np.mean(get_pscores(likelihoods_R_A_S_AC))

np.float64(19194.39339089813)

In [4]:
drbart_model_R_A_S_D = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_seconds-in-day_day-of-week/',
                     strict_parser=False)
evaluator_R_A_S_D = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D, SampleOutcomes_DRBART_Normal,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'categorical_args' : ['concept_name', 'day_of_week',
                                                                              #'(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'resources' : False,
                                                        #'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_D = evaluator_R_A_S_D.sample_cases(False, True)

100%|██████████| 1235/1235 [00:26<00:00, 46.06it/s]


In [5]:
np.mean([v.ln() for v in likelihoods_R_A_S_D[0].values()])

Decimal('-45.08902796035595916399724816')

In [6]:
np.mean(get_pscores(likelihoods_R_A_S_D))

np.float64(22641.058854674928)

In [7]:
results = {
    'drbart_model_A' : likelihoods_A,
    'drbart_model_R' : likelihoods_R,
    'drbart_model_R_A' : likelihoods_R_A,
    'drbart_model_R_A_S' : likelihoods_R_A_S,
    'drbart_model_R_A_S_AC' : likelihoods_R_A_S_AC,
    'drbart_model_R_A_S_RC' : likelihoods_R_A_S_RC,
    'drbart_model_R_A_S_RC_AC' : likelihoods_R_A_S_RC_AC,
    'drbart_model_R_A_S_RC_AC_V' : likelihoods_R_A_S_RC_AC_V,
    'drbart_model_R_A_S_D' : likelihoods_R_A_S_D,
    'drbart_model_R_A_S_D_RC_CC' : likelihoods_R_A_S_D_RC_AC
}
with open('./'+model_name+'_dr_bart_evaluation_'+log_name+'.pickle', 'wb') as handle:
    pickle.dump(results, handle)